In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import re

In [2]:
first_round = pd.read_csv("datos/classification first round/classification_final.csv")
second_round = pd.read_csv("datos/classification second_round/classification_secondRound_final.csv")

classifiers_first = pd.read_csv("datos/classification first round/classifiers_final.csv")
classifiers_second = pd.read_csv("datos/classification second_round/classifiers_secondRound_final.csv")

In [3]:
file_path = "_tempavisos.dta"

chunk_size = 10000
chunks = pd.read_stata(file_path, columns=['avisoid', 'avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo'], chunksize=chunk_size)

avisos = pd.DataFrame()

for chunk in chunks:
    avisos = pd.concat([avisos, chunk], ignore_index=True)

In [4]:
df_merged = pd.merge(first_round, second_round, on='ad_id', how='left')

## Naive with consistent taggers

Retag the second round classifications

In [5]:
# Define a mapping
classification_mapping = {
    'tot_remote': 'WFH',
    'temp_remote': 'WFH_partial',
    'semi_remote': 'WFH_partial',
    'semi_presence': 'WFH_partial',
    'not_clear_remote': 'WFH_partial',
    'not_remote': 'Not WFH'
}

# Apply the mapping
second_round['classification'] = second_round['classification'].replace(classification_mapping)

We want to detect an indicator of disagreement by classifier

In [6]:
# Step 1: For each ad, determine if the classifications agree or disagree
# We assume each ad appears exactly twice.
ad_agreement = first_round.groupby('ad_id')['classification'].nunique().reset_index()
ad_agreement['disagreement'] = ad_agreement['classification'].apply(lambda x: 0 if x == 1 else 1)

# Step 2: Merge the disagreement indicator back to the original DataFrame
df_first = first_round.merge(ad_agreement[['ad_id', 'disagreement']], on='ad_id', how='left')

# Step 3: For each classifier, compute:
#   - The total number of ads classified
#   - The sum of disagreements (i.e. the number of ads for which their classification did not match the other classifier's)
disagreement_by_classifier = df_first.groupby('classifier_id')['disagreement'].agg(total_ads='count', total_disagreements='sum')
disagreement_by_classifier['disagreement_rate'] = disagreement_by_classifier['total_disagreements'] / disagreement_by_classifier['total_ads']

In [7]:
# Step 1: For each ad, determine if the classifications agree or disagree
# We assume each ad appears exactly twice.
ad_agreement = second_round.groupby('ad_id')['classification'].nunique().reset_index()
ad_agreement['disagreement'] = ad_agreement['classification'].apply(lambda x: 0 if x == 1 else 1)

# Step 2: Merge the disagreement indicator back to the original DataFrame
df_second = second_round.merge(ad_agreement[['ad_id', 'disagreement']], on='ad_id', how='left')

# Step 3: For each classifier, compute:
#   - The total number of ads classified
#   - The sum of disagreements (i.e. the number of ads for which their classification did not match the other classifier's)
disagreement_by_classifier_second = df_second.groupby('classifier_id')['disagreement'].agg(total_ads='count', total_disagreements='sum')
disagreement_by_classifier_second['disagreement_rate'] = disagreement_by_classifier_second['total_disagreements'] / disagreement_by_classifier_second['total_ads']

Filters for each round by classifier

In [8]:
disagreement_by_classifier = disagreement_by_classifier.loc[disagreement_by_classifier['disagreement_rate'] < 0.06]

In [9]:
disagreement_by_classifier_second = disagreement_by_classifier_second.loc[disagreement_by_classifier_second['disagreement_rate'] < 0.2]

In [10]:
first_round = first_round.loc[first_round['classifier_id'].isin(disagreement_by_classifier.index)]

In [11]:
second_round = second_round.loc[second_round['classifier_id'].isin(disagreement_by_classifier_second.index)]

Now we merge them with the avisos database

In [12]:
avisos['aviso'] = avisos[['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']].apply(
    lambda row: ' '.join(row.dropna().astype(str)), axis=1
)

In [13]:
# Function to clean HTML tags
def remove_html_tags(text):
    if pd.isnull(text):
        return ""
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

# Clean and count words
avisos['aviso'] = avisos['aviso'].apply(remove_html_tags)

In [14]:
avisos = avisos[['avisoid', 'aviso']]

In [15]:
avisos = pd.merge(first_round, avisos, left_on='ad_id', right_on='avisoid', how='left')

In [17]:
avisos['workday'].value_counts()

workday
full         9356
not clear    3199
part         1845
Name: count, dtype: int64

In [19]:
avisos = avisos.loc[avisos['workday'] != 'not clear']

In [20]:
avisos['workday'].value_counts()

workday
full    9356
part    1845
Name: count, dtype: int64

In [21]:
# Step 1: Group by 'ad_id' and collect unique classifications
classification_counts = avisos.groupby('ad_id')['workday'].nunique()

# Step 2: Filter to keep only ad_ids with a single unique classification (i.e., same classification twice)
valid_ad_ids = classification_counts[classification_counts == 1].index

# Step 3: Filter the original DataFrame to keep only those ad_ids
avisos_filtered = avisos[avisos['ad_id'].isin(valid_ad_ids)]

In [22]:
avisos_filtered = avisos_filtered.drop_duplicates(subset=['aviso', 'workday'])

In [23]:
train_data = avisos_filtered[['avisoid', 'aviso', 'workday']]

In [24]:
train_data['workday'].value_counts()

workday
full    4905
part     958
Name: count, dtype: int64

In [25]:
train_data.to_csv("datos/training/naive_consistent_taggers_workday_td2.csv", index=False, encoding='utf-8')